In [2]:
import pickle
import os
import wandb
import tensorflow as tf
from wandb.keras import WandbMetricsLogger, WandbModelCheckpoint, WandbCallback
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TerminateOnNaN, CSVLogger
from tensorflow.keras import backend as K
from math import ceil
import numpy as np
from matplotlib import pyplot as plt, patches
from sklearn.model_selection import train_test_split

from models.ssd_custom_wandb import build_model
from loss_function.custom_loss import AOILoss
from performance_metrics.metrics_classification import f1
from performance_metrics.metrics_regression import mae, mse
from custom_layers.GridCenters import GridCenters

from input_encoder_decoder.input_encoder_new import SSDInputEncoder
from input_encoder_decoder.output_decoder import decode_detections
from input_encoder_decoder.data_generator import DataGenerator

%matplotlib inline
os.environ[
    "WANDB_NOTEBOOK_NAME"] = "/home/kerem/AOI/AOI-AI/implementations/ssd_models/custom_architecture/dense_training.ipynb"

In [2]:
wandb.init(
    project="aoi",

    # track hyperparameters and run metadata with wandb.config
    config={
        "conv1_filters": 32,
        "conv2_filters": 64,
        "conv3_filters": 128,
        "conv4_filters": 256,
        "conv5_filters": 512,
        "conv6_filters": 512,
        "conv1_2_kernel": (3, 3),
        "conv6_2": True,
        "detection_head_size_class": (3, 3),
        "detection_head_size_offset": (1, 1),
        "alpha": 4.0,
        "batch_size": 16,
        "max_epochs": 75,
    }
)

cfg = wandb.config

wandb: Currently logged in as: araskerem1. Use `wandb login --relogin` to force relogin


In [3]:
img_height = 256  # Height of the input images
img_width = 256  # Width of the input images
img_channels = 3  # Number of color channels of the input images
intensity_mean = 127.5  # Set this to your preference (maybe `None`). The current settings transform the input pixel values to the interval `[-1,1]`.
intensity_range = 127.5  # Set this to your preference (maybe `None`). The current settings transform the input pixel values to the interval `[-1,1]`.
n_classes = 1  # Number of positive classes
normalize_coords = True  # Whether or not the model is supposed to use coordinates relative to the image size

In [4]:
K.clear_session

model = build_model(image_size=(img_height, img_width, img_channels),
                    n_classes=n_classes,
                    l2_regularization=0.005,
                    normalize_coords=normalize_coords,
                    subtract_mean=intensity_mean,
                    divide_by_stddev=intensity_range,
                    cfg=cfg)


In [5]:
aoi_loss = AOILoss(neg_pos_ratio=3, alpha=cfg["alpha"])

steps_per_epoch = 12000 // cfg["batch_size"]
total_steps = steps_per_epoch * cfg["max_epochs"]

lr = CosineDecay(initial_learning_rate=0.001, decay_steps=total_steps, alpha=0.0001, name='cos_lr')

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss=aoi_loss.compute_loss, metrics=[f1, mae, mse])

In [6]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 identity_layer (Lambda)     (None, 256, 256, 3)          0         ['input_1[0][0]']             
                                                                                                  
 input_mean_normalization (  (None, 256, 256, 3)          0         ['identity_layer[0][0]']      
 Lambda)                                                                                          
                                                                                                  
 input_stddev_normalization  (None, 256, 256, 3)          0         ['input_mean_normalization

In [7]:
predictor_size = [model.get_layer('conv6').output_shape[1:3]]
print('Predictor Layer Dimensions: ', predictor_size)

encoder = SSDInputEncoder(img_height,
                          img_width,
                          n_classes,
                          predictor_sizes=predictor_size,
                          normalize_coords=True,
                          background_id=0)

generator = DataGenerator(parent_dir='/home/kerem/AOI/Datasets/train_pcb_crops', encoder=encoder, augmentation=True,
                          probability=0.1)

Predictor Layer Dimensions:  [(8, 8)]


In [8]:
X, y = generator.get_data()

Generating image arrays and encoding labels...
Converting images to arrays...


100%|███████████████████████████████████| 15000/15000 [00:11<00:00, 1266.82it/s]


Images as numpy:
(15000, 256, 256, 3)
Parsing ground truth labels from .csv


100%|████████████████████████████████████| 15000/15000 [00:47<00:00, 315.09it/s]


Unencoded labels:
15000
Augmenting images and relabeling...
Applying randomized augmentation...


100%|██████████████████████████████████| 15000/15000 [00:00<00:00, 21493.22it/s]


Encoded labels:
(15000, 64, 12)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

print('Train dataset: ', X_train.shape)
print('Test dataset: ', X_test.shape)
print('Train labels: ', y_train.shape)
print('Test labels: ', y_test.shape)

Train dataset:  (12000, 256, 256, 3)
Test dataset:  (3000, 256, 256, 3)
Train labels:  (12000, 64, 12)
Test labels:  (3000, 64, 12)


In [10]:
model_checkpoint = ModelCheckpoint(
    filepath='checkpoints/ssd7_epoch-{epoch:02d}_loss-{loss:.4f}_val_loss-{val_loss:.4f}.h5',
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    save_weights_only=False,
    mode='auto',
    save_freq="epoch")

csv_logger = CSVLogger(filename='ssd7_training_log.csv',
                       separator=',',
                       append=True)

early_stopping = EarlyStopping(monitor='val_loss',
                               min_delta=0.0,
                               patience=7,
                               verbose=1,
                               restore_best_weights=True)

reduce_learning_rate = ReduceLROnPlateau(monitor='val_loss',
                                         factor=0.2,
                                         patience=4,
                                         verbose=1,
                                         min_delta=0.001,
                                         cooldown=0,
                                         min_lr=0.0000001)

wandb_callback = WandbCallback(verbose=0, save_model=False)

wandb_model_checkpoint = WandbModelCheckpoint(
    filepath='saved_trained_models/custom_conv6_epoch-{epoch:02d}_val_loss-{val_loss:.4f}.keras',
    save_best_only=True)

callbacks = [  #model_checkpoint,
    #csv_logger,
    #early_stopping,
    wandb_callback
    #wandb_model_checkpoint
    #reduce_learning_rate,
]

In [11]:
history = model.fit(X_train,
                    y_train,
                    batch_size=cfg['batch_size'],
                    epochs=cfg['max_epochs'],
                    callbacks=callbacks,
                    validation_data=(X_test, y_test),
                    validation_steps=ceil(X_test.shape[0] // cfg['batch_size'])
                    #initial_epoch=initial_epoch
                    )

2023-10-05 11:05:22.109319: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2359296000 exceeds 10% of free system memory.
2023-10-05 11:05:23.719611: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2359296000 exceeds 10% of free system memory.


Epoch 1/75


2023-10-05 11:05:27.913401: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600
2023-10-05 11:05:29.369585: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7f662d189310 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2023-10-05 11:05:29.369609: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GTX 1060, Compute Capability 6.1
2023-10-05 11:05:29.394865: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:255] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2023-10-05 11:05:29.654890: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


750/750 [==============================] - 74s 87ms/step - loss: 15.7848 - f1: 0.2379 - r2: -72.0420 - val_loss: 10.8198 - val_f1: 0.4324 - val_r2: -14.0771
Epoch 2/75
750/750 [==============================] - 64s 86ms/step - loss: 9.5295 - f1: 0.5340 - r2: -9.1583 - val_loss: 8.5863 - val_f1: 0.6022 - val_r2: -6.4630
Epoch 3/75
750/750 [==============================] - 64s 86ms/step - loss: 10.5444 - f1: 0.6327 - r2: -5.6896 - val_loss: 10.8392 - val_f1: 0.6249 - val_r2: -5.4021
Epoch 4/75
750/750 [==============================] - 65s 86ms/step - loss: 8.7172 - f1: 0.6300 - r2: -4.5626 - val_loss: 7.6171 - val_f1: 0.6407 - val_r2: -3.8906
Epoch 5/75
750/750 [==============================] - 65s 86ms/step - loss: 7.9679 - f1: 0.6495 - r2: -3.4611 - val_loss: 1165.7882 - val_f1: 0.6288 - val_r2: -125.0508
Epoch 6/75
750/750 [==============================] - 65s 86ms/step - loss: 12.3918 - f1: 0.5986 - r2: -222.2760 - val_loss: 9.3809 - val_f1: 0.5931 - val_r2: -202.9588
Epoch 7/75


KeyboardInterrupt: 

In [12]:
wandb.finish()

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
f1,▁▆███▇▇▇▇▇▇▇▇▇▇
loss,█▄▅▄▃▆▄█▇▄▄▂▃▁▁
r2,▆████▁▂▃▄▄▄▅▅▅▅
val_f1,▁▇▇██▆▇▆▆▆▆▇▇▇▇
val_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁█
val_r2,████▇▇▇▇▇▇▇▇▇▇▁
best_epoch,13
best_val_loss,3.60887
epoch,14
f1,0.61781


In [ ]:
plt.figure(figsize=(8, 6))
plt.title("Loss during Training (Custom)")
plt.grid(True, linestyle='--')
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.ylim(0, 5)
plt.legend(loc='upper right', prop={'size': 15});

In [ ]:
#plt.savefig("tl_locked_ssd_loss.png")

#with open("tl_locked_ssd_loss.fig", "wb") as f:
#    pickle.dump(plt.gcf(), f)

In [ ]:
plt.figure(figsize=(8, 6))
plt.title("Custom mAP during Training (Custom)")
plt.grid(True, linestyle='--')
plt.plot(history.history['class_mAP'], label='Training')
plt.plot(history.history['val_class_mAP'], label='Validation')
plt.xlabel("Epoch")
plt.ylabel("Custom mAP")
plt.ylim(0, 1)
plt.legend(loc='lower right', prop={'size': 15});

In [ ]:
#plt.savefig("tl_locked_ssd_mAP.png")

#with open("tl_locked_ssd_mAP.fig", "wb") as f:
#    pickle.dump(plt.gcf(), f)

In [ ]:
plt.figure(figsize=(8, 6))
plt.title("Custom MAE during Training (Custom)")
plt.grid(True, linestyle='--')
plt.plot(history.history['offset_MAE'], label='Training')
plt.plot(history.history['val_offset_MAE'], label='Validation')
plt.xlabel("Epoch")
plt.ylabel("Custom MAE")
plt.ylim(0, 0.15)
plt.legend(loc='upper right', prop={'size': 15});

In [ ]:
#plt.savefig("tl_locked_ssd_MAE.png")

#with open("tl_locked_ssd_MAE.fig", "wb") as f:
#    pickle.dump(plt.gcf(), f)

In [ ]:
predictions = model.predict(X_test[:50])
print(predictions.shape)

In [ ]:
decoded_pred = decode_detections(predictions, img_height=img_height, img_width=img_width)
print(decoded_pred[0])

In [ ]:
decoded_labels = decode_detections(y_test[:50], img_height=img_height, img_width=img_width)
print(decoded_labels[0])

In [ ]:
for i in range(len(decoded_pred[:20])):
    plt.figure(figsize=(10, 6))
    plt.imshow(X_test[i])
    current_axis = plt.gca()

    colors = plt.cm.hsv(np.linspace(0, 1, n_classes + 1)).tolist()  # Set the colors for the bounding boxes
    classes = ['background', 'ic']  # Just so we can print class names onto the image instead of IDs

    for label in decoded_labels[i]:
        pred = np.array(label)
        corners = np.reshape(pred[2:], (-1, 2)).astype(int)
        color = colors[int(pred[0])]
        label = '{}'.format(classes[int(pred[0])])
        for points in corners:
            current_axis.add_patch(plt.Circle(tuple(points), 2, fill=False, color='blue'))

    for label in decoded_pred[i]:
        pred = np.array(label)
        corners = np.reshape(pred[2:], (-1, 2)).astype(int)
        color = colors[int(pred[0])]
        label = '{}: {:.2f}'.format(classes[int(pred[0])], pred[1])
        for points in corners:
            current_axis.add_patch(plt.Circle(tuple(points), 2, fill=False, color='red'))
        center_x = (corners[0, 0] + corners[3, 0]) / 2
        center_y = (corners[0, 1] + corners[3, 1]) / 2
        current_axis.text(center_x, center_y, label, size='x-small', color='white',
                          bbox={'facecolor': color, 'alpha': 1.0})


In [ ]:
#model.save("saved_trained_models/custom_realistic_pcb_conv6_256_15000_relu_noise")